# Ноутбук 01 — Предобработка данных и конструирование признаков
**Подраздел 2.1 ПЗ — Описание и предобработка данных**

Запускать после `00_eda_dataset_overview.ipynb`.

Артефакты:
- `data/interim/train_clean.parquet` — очищенная объединённая таблица (дневной гранулярности)
- `data/interim/weekly_sales.parquet` — недельный ряд суммарных продаж (используется в 02)
- `data/processed/features_train.parquet` — обучающая выборка с лаговыми и скользящими признаками
- `data/processed/features_test.parquet` — тестовая выборка с теми же признаками
- `reports/tables/table_2_1_missing_values.csv` — таблица пропущенных значений до и после очистки

## Ячейка 0 — Импорты и конфигурация

In [ ]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from src.config import (
    DATA_INT, DATA_PROC,
    DATE_COL, STORE_COL, FAMILY_COL, GROUP_KEYS,
    TARGET, TRAIN_CUTOFF,
    LAG_WEEKS, ROLLING_WINDOWS,
    TABLES,
)
from src.io.preprocess import (
    load_raw_files,
    fill_oil_gaps,
    remove_duplicates,
    save_interim,
)

# Создать директории, если отсутствуют
DATA_INT.mkdir(parents=True, exist_ok=True)
DATA_PROC.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("Конфигурация загружена.")
print(f"  TRAIN_CUTOFF   : {TRAIN_CUTOFF}")
print(f"  LAG_WEEKS      : {LAG_WEEKS}")
print(f"  ROLLING_WINDOWS: {ROLLING_WINDOWS}")

## Ячейка 1 — Загрузка сырых данных

In [ ]:
data = load_raw_files()

train        = data["train"]
stores       = data["stores"]
oil          = data["oil"]
holidays     = data["holidays"]
transactions = data["transactions"]

print("Загружено строк:")
for name, df in data.items():
    print(f"  {name:15s}: {len(df):>10,} строк  |  {df.shape[1]} столбцов")

## Ячейка 2 — Заполнение пропусков в ценах на нефть

Пропуски возникают в выходные дни, поскольку торги на нефтяном рынке
в эти дни не проводятся. Вследствие этого применяется метод прямого
заполнения (`ffill`), а оставшиеся граничные значения — обратного (`bfill`).

In [ ]:
n_missing_before = oil["dcoilwtico"].isna().sum()
oil_clean = fill_oil_gaps(oil)
n_missing_after = oil_clean["dcoilwtico"].isna().sum()

print(f"Пропуски в dcoilwtico до очистки : {n_missing_before}")
print(f"Пропуски в dcoilwtico после очистки: {n_missing_after}")

## Ячейка 3 — Удаление дублей и отрицательных продаж

Отрицательные значения `sales` интерпретируются как возвраты товара.
Поскольку целевая переменная — суммарные продажи, значения ниже нуля
усекаются до нуля (`clip(lower=0)`).

In [ ]:
train = remove_duplicates(train)

n_negative = (train["sales"] < 0).sum()
train["sales"] = train["sales"].clip(lower=0)
print(f"Отрицательных значений sales (усечено до 0): {n_negative}")

## Ячейка 4 — Слияние всех таблиц

Объединение выполняется последовательно:
1. `train` + `stores` — по ключу `store_nbr`
2. результат + `oil_clean` — по ключу `date`
3. результат + `transactions` — по ключам `date`, `store_nbr`
4. результат + `holidays` (только национальные) — по ключу `date`

Для праздников используется флаг `is_holiday` (0 / 1).

**Исправление:** после left-merge с `transactions` столбец `transactions`
содержит NaN для дат без записи о транзакциях (магазин закрыт, выходной).
Вследствие этого NaN обнуляются явно до построения лаговых признаков.

In [ ]:
# Шаг 1: train + stores
df = train.merge(stores, on=STORE_COL, how="left")

# Шаг 2: + oil
oil_clean[DATE_COL] = pd.to_datetime(oil_clean[DATE_COL])
df = df.merge(oil_clean[[DATE_COL, "dcoilwtico"]], on=DATE_COL, how="left")

# Шаг 3: + transactions
transactions[DATE_COL] = pd.to_datetime(transactions[DATE_COL])
df = df.merge(transactions, on=[DATE_COL, STORE_COL], how="left")
# FIX-1: не каждый магазин имеет запись транзакций на каждую дату;
# left-merge порождает NaN в столбце transactions.
# Семантически 0 транзакций — закрытый магазин / выходной день.
df["transactions"] = df["transactions"].fillna(0).astype(int)

# Шаг 4: + holidays (только национальные, без перенесённых)
holidays[DATE_COL] = pd.to_datetime(holidays[DATE_COL])
nat_holidays = (
    holidays[
        (holidays["locale"] == "National") &
        (holidays["transferred"] == False)
    ][[DATE_COL]]
    .drop_duplicates()
    .assign(is_holiday=1)
)
df = df.merge(nat_holidays, on=DATE_COL, how="left")
df["is_holiday"] = df["is_holiday"].fillna(0).astype(int)

print(f"Итоговая таблица: {df.shape[0]:,} строк  |  {df.shape[1]} столбцов")
print(f"Диапазон дат: {df[DATE_COL].min().date()} — {df[DATE_COL].max().date()}")

# Контрольная проверка: NaN в transactions после исправления
n_nan_tx = df["transactions"].isna().sum()
print(f"NaN в столбце transactions после fillna(0): {n_nan_tx}  — {'ОК' if n_nan_tx == 0 else 'ОШИБКА'}")

## Ячейка 5 — Таблица пропущенных значений (Таблица 2.1 ПЗ)

Таблица строится **после слияния**, чтобы зафиксировать исходное
количество NaN в столбце `transactions` до их обнуления.

In [ ]:
def missing_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    total = df.isna().sum()
    pct   = (total / len(df) * 100).round(2)
    return pd.DataFrame({
        "Таблица"   : name,
        "Столбец"   : total.index,
        "Пропусков" : total.values,
        "Доля, %"   : pct.values,
    })

# Таблица строится после слияния по итоговому df
missing_parts = [
    missing_summary(train,        "train"),
    missing_summary(oil_clean,    "oil"),
    missing_summary(transactions, "transactions"),
    missing_summary(df,           "merged_df"),
]
missing_df = pd.concat(missing_parts, ignore_index=True)
missing_df = missing_df[missing_df["Пропусков"] > 0]

missing_df.to_csv(TABLES / "table_2_1_missing_values.csv", index=False)
print("Таблица 2.1 сохранена.")
display(missing_df)

## Ячейка 6 — Сохранение очищенной таблицы в `data/interim/`

In [ ]:
save_interim(df, "train_clean")
# Результат: data/interim/train_clean.parquet

## Ячейка 7 — Недельная агрегация суммарных продаж

Ряд агрегируется по всем магазинам и семействам товаров.
Точка отсчёта недели — понедельник (`W-MON`).
Результат используется в ноутбуке 02 для анализа стационарности.

In [ ]:
weekly = (
    df.groupby(DATE_COL)["sales"]
    .sum()
    .resample("W-MON")
    .sum()
    .rename("sales")
)

weekly_path = DATA_INT / "weekly_sales.parquet"
weekly.to_frame().to_parquet(weekly_path)

print(f"Недельный ряд: {len(weekly)} наблюдений")
print(f"  от {weekly.index[0].date()} до {weekly.index[-1].date()}")
print(f"Сохранено: {weekly_path}")

## Ячейка 8 — Конструирование временны́х признаков

Для моделей машинного обучения (XGBoost, случайный лес, эластичная сеть)
из столбца `date` извлекаются календарные признаки.

In [ ]:
df["year"]        = df[DATE_COL].dt.year
df["month"]       = df[DATE_COL].dt.month
df["week"]        = df[DATE_COL].dt.isocalendar().week.astype(int)
df["day_of_week"] = df[DATE_COL].dt.dayofweek
df["day_of_year"] = df[DATE_COL].dt.dayofyear

# Синусоидальное и косинусоидальное кодирование недели года (цикличность)
df["week_sin"] = np.sin(2 * np.pi * df["week"] / 52)
df["week_cos"] = np.cos(2 * np.pi * df["week"] / 52)

print("Временны́е признаки добавлены:", ["year","month","week","day_of_week","day_of_year","week_sin","week_cos"])

## Ячейка 9 — Лаговые и скользящие признаки

Признаки строятся внутри каждой группы `(store_nbr, family)`,
чтобы не допустить утечки данных между рядами разных магазинов.

Лаги и окна задаются в `src/config.py`:
- `LAG_WEEKS = [1, 2, 4, 12, 52]`
- `ROLLING_WINDOWS = [4, 12]`

**Исправление:** для скользящего стандартного отклонения установлен
`min_periods=2`, поскольку при `n=1` стандартное отклонение математически
не определено и pandas возвращает NaN. Оставшиеся NaN (первая строка
каждой группы) обнуляются через `fillna(0)`.

In [ ]:
df = df.sort_values([STORE_COL, FAMILY_COL, DATE_COL])

# Лаговые признаки
for lag in LAG_WEEKS:
    col_name = f"sales_lag_{lag}w"
    df[col_name] = (
        df.groupby(GROUP_KEYS)["sales"]
        .shift(lag)
    )

# Скользящие статистики — среднее и стандартное отклонение
# FIX-2: mean использует min_periods=1 (нет NaN кроме граничных),
#        std использует min_periods=2, так как при n=1 std=NaN по определению.
#        Остаточные NaN (первая строка группы у std) обнуляются.
for window in ROLLING_WINDOWS:
    # Среднее
    col_mean = f"sales_roll_{window}w_mean"
    df[col_mean] = (
        df.groupby(GROUP_KEYS)["sales"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    # Стандартное отклонение
    col_std = f"sales_roll_{window}w_std"
    df[col_std] = (
        df.groupby(GROUP_KEYS)["sales"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=2).std())
        .fillna(0)
    )

lag_cols = [c for c in df.columns if c.startswith("sales_lag_") or c.startswith("sales_roll_")]
print(f"Лаговых и скользящих признаков добавлено: {len(lag_cols)}")
print("  ", lag_cols)

## Ячейка 10 — Разбиение на обучающую и тестовую выборки

Граница `TRAIN_CUTOFF = "2017-06-01"` задана в `src/config.py`.
Строки с `date < TRAIN_CUTOFF` образуют обучающую выборку,
строки с `date >= TRAIN_CUTOFF` — тестовую.

In [ ]:
cutoff = pd.Timestamp(TRAIN_CUTOFF)

df_train = df[df[DATE_COL] < cutoff].copy()
df_test  = df[df[DATE_COL] >= cutoff].copy()

print(f"Обучающая выборка : {len(df_train):>10,} строк  ({df_train[DATE_COL].min().date()} — {df_train[DATE_COL].max().date()})")
print(f"Тестовая выборка  : {len(df_test):>10,} строк  ({df_test[DATE_COL].min().date()} — {df_test[DATE_COL].max().date()})")

## Ячейка 11 — Сохранение финальных датасетов в `data/processed/`

In [ ]:
train_path = DATA_PROC / "features_train.parquet"
test_path  = DATA_PROC / "features_test.parquet"

df_train.to_parquet(train_path, index=False)
df_test.to_parquet(test_path,  index=False)

print(f"Сохранено: {train_path}")
print(f"Сохранено: {test_path}")

## Ячейка 12 — Контрольная проверка

Финальная проверка целостности данных перед запуском ноутбука 02.

In [ ]:
print("=" * 55)
print("КОНТРОЛЬНАЯ ПРОВЕРКА")
print("=" * 55)

# Недельный ряд
print(f"\nНедельный ряд: {len(weekly)} наблюдений")
print(weekly.describe().to_string())

# Пропуски в лаговых признаках (норма — первые lag строк группы)
lag_na = df_train[lag_cols].isna().sum()
print(f"\nПропуски в лаговых/скользящих признаках (обучающая выборка):")
print(lag_na[lag_na > 0].to_string())

# Отрицательных продаж нет
assert (df_train["sales"] < 0).sum() == 0, "Обнаружены отрицательные продажи!"
assert (df_test["sales"] < 0).sum()  == 0, "Обнаружены отрицательные продажи!"
print("\nОтрицательных продаж: 0  — ОК")

# FIX-1 проверка: NaN в transactions недопустимы
assert df_train["transactions"].isna().sum() == 0, "NaN в transactions (обучающая выборка)!"
assert df_test["transactions"].isna().sum()  == 0, "NaN в transactions (тестовая выборка)!"
print("NaN в transactions: 0  — ОК")

# FIX-2 проверка: NaN в скользящих std недопустимы
std_cols = [c for c in lag_cols if c.endswith("_std")]
std_na = df_train[std_cols].isna().sum().sum()
assert std_na == 0, f"NaN в скользящих std: {std_na}"
print("NaN в скользящих std: 0  — ОК")

print("=" * 55)
print("Предобработка завершена. Запустите 02_stationarity_acf_pacf_adf.ipynb")